# AXI test 18: the serving side a fifth time, with the grammar of the reply chosen by the type of the request
**Why this test exists.** In the fourth run (test 17) one retry did not raise the exactness on jobs nobody had seen (98.1 % with it, 98.1 % without). Eight jobs in the first 128 slipped, every first slip on an **Add** request: five wrote an answer in the wrong *shape* (a service written as an edge, an extra delete in front of an add), three wrote the right shape with a wrong *number*. This run does not ask again. It changes what the engine is **allowed to write for each request**: a grammar chosen by the kind of the request (`kind`), a grammar that also names the coordinates the request names (`named`), and your anchor line (concept-map item 110) at the top and the end of the prompt, or at the end only. It runs seven cells of 64 jobs: `F0` (control), `FK`, `FKA`, `FKT` and `FN` on jobs 128 to 191, which no run has seen, and `SK` and `SN` on jobs 0 to 63, where the slips were read. The kept chat is not run, and there is **no retry in any cell**. The predictions and the reading rules were written before this run in `docs/PROVIDER_SIDE_TYPED_TEST.md`.

Model: **Qwen2.5-7B-Instruct, 4-bit AWQ**, engine **vLLM 0.31.0**, cache fixed at **80,000 tokens**, GPU **T4**.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. **Choose the T4 even if your plan offers a faster GPU:** every earlier number is from a T4, and another GPU would make the comparison with them unreliable. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. **Keep this browser tab open** while a cell runs.

**How long it takes (a guess from the earlier runs):** installing the engine about 5 minutes, loading the model 5 to 10 minutes, the safety check 3 minutes, the seven cells about 25 minutes. In all about 45 to 55 minutes.

**If Colab stops** (the tab closes, the GPU is taken away): reconnect, run Cells 1 to 5 again (do not forget Cell 4b), and then run the cell you were in. Finished cells are skipped; a cell that was cut off starts again from its beginning (the longest takes about 6 minutes).

Do not paste any keys anywhere.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: get the code from GitHub. If it stops with 'the code is not on GitHub yet', finish the update 64 commands in Codespaces (git push) and run this cell again

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/provider_side_typed.py"), "the code is not on GitHub yet: finish the update 64 commands (git push) first"
assert os.path.exists("/content/axi/python/axi/experiments/q_measure.py"), "the code is not on GitHub yet: finish the update 64 commands (git push) first"
assert os.path.exists("/content/axi/docs/results/axi_provider_bounded.json"), "the third run's file is not on GitHub"
print("code ok")

### Cell 3: install the engine, version 0.31.0 (about 5 minutes). If Colab says 'Restart session', press it, then run Cells 2, 3, 4 and 5 again (Cell 3 will then be quick)

In [ ]:
!pip -q install vllm==0.31.0

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). If it prints 'Drive NOT connected', the run still works but nothing survives a disconnect

In [ ]:
import sys, os, json, time, glob, re
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_provider_typed")

### Cell 4b: fix the output stream (about 1 second). vLLM 0.31.0 asks the notebook for a file number of its output and a Colab cell has none, so Cell 5 stops with `io.UnsupportedOperation: fileno`. This cell gives the notebook's output the usual numbers 1 and 2. Run it once after every restart, before Cell 5

In [ ]:
import sys
for name, fd in (("stdout", 1), ("stderr", 2)):
    s = getattr(sys, name)
    try: s.fileno()
    except Exception: s.fileno = (lambda fd=fd: fd)
print("output numbers:", sys.stdout.fileno(), sys.stderr.fileno())

### Cell 5: load the model and the engine with one expression per request (downloads about 5.5 GB once; about 5 to 10 minutes). It prints the engine's own numbers. If it fails with a red error, stop and send me the red text (if the red text says `fileno`, run Cell 4b first; if it says `StructuredOutputsParams`, send it to me and do nothing else)

In [ ]:
from axi.experiments import provider_side_typed as T
from axi.experiments import q_measure as Q
eng = T.TypedVLLMEngine()
info = eng.info(); print(json.dumps({k: v for k, v in info.items() if k != "constraint"}, indent=1))
print("constraint:", info["constraint"]["kind"], "base expression with a name of at most", info["constraint"]["name_cap"], "characters", info["constraint"]["regex_sha256"][:16], "(each request carries its own expression)")
eng.warm_up()
print("cache size the engine will use:", info["capacity_tokens"], "tokens (", info["capacity_source"], ")")

def go(label):
    """Runs one cell unless its file is already there; saves it and copies it to Drive. A cell that was cut off starts again."""
    name = f"axi_provider_t_{label}.json"; K, first, prompt, shape = T.CELLS[label]
    print(f"=== {label}: jobs {first} to {first + K - 1}, prompt {prompt}, grammar {shape}: {store.restore(name)}")
    if os.path.exists(store.path(name)): print("   already finished, skipping"); return
    t0 = time.time()
    def progress(i): print(f"   round {i}   {time.time() - t0:6.0f} s", end="\r")
    c = T.run_cell(eng, label, progress=progress)
    T.save_cell(c, store.path(name)); store.persist(name)
    s = T.summarize_cell4(c); f = lambda x, d=1: "-" if x is None else f"{x:.{d}f}"
    print(f"\n   done: {s['steps_done']} steps, {s['wall_seconds']:.0f} s, {f(s['seconds_per_step'], 3)} s/step, computed prompt tokens per step {f(s['computed_per_step'], 0)}, exact steps {s['exact_share'] * 100:.1f} %, jobs exact to the end {s['jobs_exact_to_end']}, outside the language {s['out_of_language']}, refused by the strict parser {s['unparsed']}, {s['distinct_regexes']} expressions, exact steps per hour {f(s['exact_per_hour'], 0)}")

### Cell 6: SAFETY CHECK (about 3 minutes). 3 jobs of 3 steps with the heaviest variant (the anchor prompt and the grammar that names the coordinates), then the PROBE at the eight steps where the earlier runs slipped: the prompt of each step is rebuilt and asked under the capped, kind, named, kind-with-anchor and kind-with-tail variants, and once with a PARITY turn that says what the request names. It prints what the engine answers. **A wrong answer is not a STOP** (that is what the test is for). If it says STOP, do not run the next cells: send me the printout

In [ ]:
from axi.experiments.agent_loop import strict_parse
smoke = T.run_jobs(eng, [0, 1, 2], "anchor", "named", steps=3)
for x in smoke["sessions"][0]["log"]: print(f"step {x['t']}: prompt {x['n_prompt']:5d}  cached {x['n_cached']}  new {x['n_new']:3d}  {x['text'][:50]!r}")
allrep = [(x["parsed"], x["n_cached"], x["in_language"], x["n_new"]) for s in smoke["sessions"] for x in s["log"]]
ok_cached = all(c is not None for _, c, _, _ in allrep)
ok_parse = all(p for p, _, _, _ in allrep)
ok_lang = all(l for _, _, l, _ in allrep)
probe = T.known_slip_probe(eng)
shape_of = {name: shape for name, _, shape in T.PROBE_VARIANTS}; shape_of["parity"] = "capped"
print()
for p in probe:
    print(f"job {p['job']}, step {p['step']} ({p['kind']}): {p['instruction']}")
    print(f"   the earlier reply was {p['earlier']!r} (inside the kind language: {p['earlier_in_kind']}, inside the named language: {p['earlier_in_named']}); the oracle wrote {p['oracle']!r}")
    for k, v in p["replies"].items(): print(f"   {k:10s} {v!r}{'   = the oracle' if v.strip() == p['oracle'] else ''}")
    print(f"   the parity turn was: {p['parity']!r}")
probe_parse = all(strict_parse(v) is not None for p in probe for v in p["replies"].values())
probe_lang = all(T.in_language(shape_of[k], p["instruction"], v) for p in probe for k, v in p["replies"].items())
probe_long = any(n >= 96 for p in probe for n in p["n_new"].values())
probe_cached = all(c is not None for p in probe for c in p["cached"].values())
print()
print("STOP: this vLLM version does not report cached tokens. Send me the printout and the version above." if not ok_cached or not probe_cached else
      "STOP: a reply of the safety cell was refused by the strict parser, so the constraint is not working. Send me the printout." if not ok_parse else
      "STOP: a reply of the safety cell is outside the expression of its own request, so one expression per request is not working. Send me the printout." if not ok_lang else
      "STOP: a reply of the probe is refused by the strict parser. Send me the printout." if not probe_parse else
      "STOP: a reply of the probe is outside the expression of its own request. Send me the printout." if not probe_lang else
      "STOP: a reply of the probe used all 96 new tokens. Send me the printout." if probe_long else
      "OK: cached counts are reported, every reply is well formed and inside the expression of its request, and the probe's replies end. You can go on.")

### Cell 7: the seven cells, in this order: `F0`, `FK`, `FKA`, `FKT`, `FN` (jobs 128 to 191: control, kind grammar, kind grammar with the anchor at the top and the end, with the anchor at the end only, kind and coordinates), then `SK` and `SN` (jobs 0 to 63, kind and kind-and-coordinates). About 25 minutes in all. A counter shows the round

In [ ]:
for label in T.ORDER: go(label)

### Cell 8: put the finished cells together, read the third run's file from the repository, print the table, the first misses, the 17 reading-rule lines and Q = O / M of every cell, and download your file. You can run it at any time, with whatever cells are finished

In [ ]:
from google.colab import files
THIRD = "/content/axi/docs/results/axi_provider_bounded.json"
paths = sorted(glob.glob(store.path("axi_provider_t_*.json")))
print(len(paths), "cells:", [os.path.basename(p) for p in paths])
m = T.merge_files(paths, store.path("axi_provider_typed.json"), third_run=THIRD); store.persist("axi_provider_typed.json")
old = T.load_third_run(THIRD)["cells"]
S = T.summarize_all(m["cells"]); T.print_cells(S); T.print_misses(S); T.print_verdict(T.verdict_typed(m["cells"], old))
print(); print(Q.render_table({"Q = O / M of the cells of this run": [Q.q_row(c) for c in m["cells"]]}))
files.download(store.path("axi_provider_typed.json"))

### Done
File downloaded: `axi_provider_typed.json`. Upload it to the chat, with the cell files from your Drive folder `axi_provider_typed` (the `axi_provider_t_*.json` files), a screenshot of the printed table and of the 17 reading-rule lines, the printout of the probe (Cell 6), and the engine numbers printed by Cell 5 (vLLM version, GPU, cache size). Do not paste any keys anywhere.